# Phase 4 — Emotion Detection Model

In [1]:
import os
import pickle
import time
import numpy as np
import torch
import torch.nn as nn
import torch.optim as optim
from torch.utils.data import Dataset, DataLoader
from torchvision import transforms, models
from PIL import Image
from sklearn.metrics import classification_report

## Configuration

In [ ]:
#Paths
DATA_DIR    = os.path.expanduser('~/Desktop/2026-Sem1/COS30082-Applied Machine Learning/emotional_detection/fer_2013')
OUTPUT_PATH = 'emotion_model.pkl'

EMOTIONS   = ['angry', 'disgust', 'fear', 'happy', 'neutral', 'sad', 'surprise']
IMG_SIZE   = 48
BATCH_SIZE = 64
EPOCHS     = 25
LR         = 0.001   # 0.001
DEVICE = torch.device('mps' if torch.backends.mps.is_available() else 'cpu')

print(f"Device : {DEVICE}")
print(f"Data   : {DATA_DIR}")

Device : mps
Data   : /Users/ronuts/Desktop/2026-Sem1/COS30082-Applied Machine Learning/emotional_detection/fer_2013


## Dataset

In [3]:
class FER2013Folder(Dataset):
    def __init__(self, root, transform=None):
        self.samples   = []
        self.transform = transform
        for label, emotion in enumerate(EMOTIONS):
            folder = os.path.join(root, emotion)
            if not os.path.isdir(folder):
                continue
            for fname in os.listdir(folder):
                if fname.lower().endswith(('.jpg', '.jpeg', '.png')):
                    self.samples.append((os.path.join(folder, fname), label))

    def __len__(self):
        return len(self.samples)

    def __getitem__(self, idx):
        path, label = self.samples[idx]
        img = Image.open(path).convert('RGB')
        if self.transform:
            img = self.transform(img)
        return img, label

## Transforms

In [4]:
train_transform = transforms.Compose([
    transforms.Resize((IMG_SIZE, IMG_SIZE)),
    transforms.RandomHorizontalFlip(),
    transforms.RandomRotation(10),
    transforms.ColorJitter(brightness=0.2, contrast=0.2),
    transforms.ToTensor(),
    transforms.Normalize([0.485, 0.456, 0.406],
                         [0.229, 0.224, 0.225])
])

val_transform = transforms.Compose([
    transforms.Resize((IMG_SIZE, IMG_SIZE)),
    transforms.ToTensor(),
    transforms.Normalize([0.485, 0.456, 0.406],
                         [0.229, 0.224, 0.225])
])

## Model — MobileNetV2 Backbone

In [5]:
class EmotionCNN(nn.Module):
    def __init__(self, num_classes=7, dropout=0.4):
        super().__init__()
        backbone = models.mobilenet_v2(weights=models.MobileNet_V2_Weights.DEFAULT)
        backbone.classifier = nn.Sequential(
            nn.Dropout(dropout),
            nn.Linear(backbone.last_channel, 256),
            nn.ReLU(),
            nn.Dropout(dropout),
            nn.Linear(256, num_classes)
        )
        self.model = backbone

    def forward(self, x):
        return self.model(x)

## Train & Eval Helpers

In [6]:
def train_epoch(model, loader, criterion, optimizer, scaler):
    model.train()
    total_loss, correct, total = 0.0, 0, 0
    for imgs, labels in loader:
        imgs, labels = imgs.to(DEVICE), labels.to(DEVICE)
        optimizer.zero_grad()
        with torch.amp.autocast('cuda', enabled=DEVICE.type == 'cuda'):
            out  = model(imgs)
            loss = criterion(out, labels)
        scaler.scale(loss).backward()
        scaler.step(optimizer)
        scaler.update()
        total_loss += loss.item() * imgs.size(0)
        correct    += (out.argmax(1) == labels).sum().item()
        total      += imgs.size(0)
    return total_loss / total, correct / total


def eval_epoch(model, loader, criterion):
    model.eval()
    total_loss, correct, total = 0.0, 0, 0
    all_preds, all_labels = [], []
    with torch.no_grad():
        for imgs, labels in loader:
            imgs, labels = imgs.to(DEVICE), labels.to(DEVICE)
            out   = model(imgs)
            loss  = criterion(out, labels)
            preds = out.argmax(1)
            total_loss += loss.item() * imgs.size(0)
            correct    += (preds == labels).sum().item()
            total      += imgs.size(0)
            all_preds.extend(preds.cpu().numpy())
            all_labels.extend(labels.cpu().numpy())
    return total_loss / total, correct / total, all_preds, all_labels

## Load Data

In [7]:
train_ds = FER2013Folder(os.path.join(DATA_DIR, 'train'), transform=train_transform)
val_ds   = FER2013Folder(os.path.join(DATA_DIR, 'test'),  transform=val_transform)

print(f"Train samples : {len(train_ds)}")
print(f"Val   samples : {len(val_ds)}")

train_loader = DataLoader(train_ds, batch_size=BATCH_SIZE, shuffle=True,
                          num_workers=0, pin_memory=False)
val_loader   = DataLoader(val_ds,   batch_size=BATCH_SIZE, shuffle=False,
                          num_workers=0, pin_memory=False)

Train samples : 28709
Val   samples : 7178


## Initialize Model, Optimizer & Scheduler

In [8]:
model     = EmotionCNN(num_classes=len(EMOTIONS)).to(DEVICE)
criterion = nn.CrossEntropyLoss(label_smoothing=0.1)
optimizer = optim.AdamW(model.parameters(), lr=LR, weight_decay=1e-4)
scheduler = optim.lr_scheduler.CosineAnnealingLR(optimizer, T_max=EPOCHS)
scaler    = torch.amp.GradScaler('cpu', enabled=DEVICE.type == 'cpu')

history    = {'train_loss': [], 'train_acc': [], 'val_loss': [], 'val_acc': []}
best_acc   = 0.0
best_state = None

## Training Loop

In [9]:
from tqdm import tqdm

for epoch in range(1, EPOCHS + 1):
    t0 = time.time()
    tr_loss, tr_acc             = train_epoch(model, train_loader, criterion, optimizer, scaler)
    va_loss, va_acc, preds, lbs = eval_epoch(model, val_loader, criterion)
    scheduler.step()

    history['train_loss'].append(round(tr_loss, 4))
    history['train_acc'].append(round(tr_acc,   4))
    history['val_loss'].append(round(va_loss,   4))
    history['val_acc'].append(round(va_acc,     4))

    flag = ''
    if va_acc > best_acc:
        best_acc   = va_acc
        best_state = {k: v.cpu().clone() for k, v in model.state_dict().items()}
        flag = '  ★ best'

    elapsed = time.time() - t0
    remaining = elapsed * (EPOCHS - epoch)
    mins, secs = divmod(int(remaining), 60)
    print(f"Epoch {epoch:02d}/{EPOCHS}  "
          f"train_loss={tr_loss:.4f}  train_acc={tr_acc:.4f}  "
          f"val_loss={va_loss:.4f}  val_acc={va_acc:.4f}  "
          f"[{elapsed:.1f}s]  ETA: {mins}m {secs}s{flag}")

Epoch 01/25  train_loss=1.5944  train_acc=0.4240  val_loss=1.4178  val_acc=0.5174  [36.3s]  ★ best
Epoch 02/25  train_loss=1.4121  train_acc=0.5237  val_loss=1.3279  val_acc=0.5605  [26.1s]  ★ best
Epoch 03/25  train_loss=1.3518  train_acc=0.5582  val_loss=1.2932  val_acc=0.5789  [25.8s]  ★ best
Epoch 04/25  train_loss=1.3131  train_acc=0.5814  val_loss=1.2861  val_acc=0.5946  [24.9s]  ★ best
Epoch 05/25  train_loss=1.2825  train_acc=0.5948  val_loss=1.2373  val_acc=0.6134  [24.9s]  ★ best
Epoch 06/25  train_loss=1.2509  train_acc=0.6116  val_loss=1.2452  val_acc=0.6117  [24.1s]
Epoch 07/25  train_loss=1.2293  train_acc=0.6221  val_loss=1.2232  val_acc=0.6186  [23.5s]  ★ best
Epoch 08/25  train_loss=1.2004  train_acc=0.6411  val_loss=1.2158  val_acc=0.6294  [23.7s]  ★ best
Epoch 09/25  train_loss=1.1788  train_acc=0.6549  val_loss=1.2083  val_acc=0.6226  [24.4s]
Epoch 10/25  train_loss=1.1511  train_acc=0.6666  val_loss=1.2161  val_acc=0.6259  [22.1s]
Epoch 11/25  train_loss=1.1273  tr

## Evaluation & Save PKL

In [10]:
model.load_state_dict(best_state)
_, best_val_acc, final_preds, final_labels = eval_epoch(model, val_loader, criterion)

report = classification_report(final_labels, final_preds,
                                target_names=EMOTIONS, output_dict=True)

print(f"Best Val Accuracy : {best_val_acc:.4f}\n")
print(classification_report(final_labels, final_preds, target_names=EMOTIONS))

payload = {
    'model_state_dict'      : best_state,
    'model_class'           : 'EmotionCNN',
    'emotions'              : EMOTIONS,
    'img_size'              : IMG_SIZE,
    'normalize_mean'        : [0.485, 0.456, 0.406],
    'normalize_std'         : [0.229, 0.224, 0.225],
    'training_history'      : history,
    'classification_report' : report,
    'phase'                 : 'emotion_detection',
}

with open(OUTPUT_PATH, 'wb') as f:
    pickle.dump(payload, f)

print(f"\nemotion_model.pkl saved -> {OUTPUT_PATH}")

Best Val Accuracy : 0.6556

              precision    recall  f1-score   support

       angry       0.59      0.56      0.58       958
     disgust       0.65      0.56      0.60       111
        fear       0.53      0.49      0.51      1024
       happy       0.85      0.85      0.85      1774
     neutral       0.59      0.63      0.61      1233
         sad       0.52      0.53      0.53      1247
    surprise       0.77      0.78      0.77       831

    accuracy                           0.66      7178
   macro avg       0.64      0.63      0.64      7178
weighted avg       0.66      0.66      0.65      7178


emotion_model.pkl saved -> emotion_model.pkl
